In [1]:
# Run on A100

In [2]:
from experiments import Mesh3D, discretize, continuous_coefficient_3d
import numpy as np
import time
import pandas as pd
import gc
import torch
import os
import json
import pyamgx
import tqdm

AMGX version 2.5.0
Built on Jul  5 2025, 09:37:04
Compiled with CUDA Runtime 12.6, using CUDA driver 12.7
The AMGX_initialize_plugins API call is deprecated and can be safely removed.


In [3]:
all_meshes = [Mesh3D.unit_cube_uniform(1, 1, 1)]
mappings = []

for i in range(1, 8):
    mesh, mapping = all_meshes[-1].refine(1)
    all_meshes.append(mesh)
    mappings.append(mapping)

In [4]:
def test_config(cfg, problem, rep_setup=3, rep_solve=5):
    rsc = pyamgx.Resources().create_simple(cfg)

    A = pyamgx.Matrix().create(rsc)
    b = pyamgx.Vector().create(rsc)
    x = pyamgx.Vector().create(rsc)

    A.upload_CSR(problem.exact_form_matrix)
    b.upload(problem.load_vector)

    solver = pyamgx.Solver().create(rsc, cfg)

    setup_times = []
    for _ in range(rep_setup):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        solver.setup(A)
        end.record()
        torch.cuda.synchronize()
        setup_times.append(start.elapsed_time(end) / 1000)

    solve_times = []
    for _ in range(rep_solve):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        x.set_zero(n=problem.load_vector.shape[0], block_dim=1)
        start.record()
        solver.solve(b, x)
        end.record()
        torch.cuda.synchronize()
        solve_times.append(start.elapsed_time(end) / 1000)

    x_vec = np.zeros_like(problem.load_vector)
    x.download(x_vec)
    residual = np.linalg.norm(problem.exact_form_matrix @ x_vec - problem.load_vector)

    A.destroy()
    b.destroy()
    x.destroy()
    solver.destroy()
    rsc.destroy()

    return {
        "setup_time": min(setup_times),
        "solve_time": min(solve_times),
        "residual": residual,
    }

In [5]:
def patch_config(cfg):
    cfg = cfg.copy()
    cfg["solver"]["print_grid_stats"] = 0
    cfg["solver"]["print_solve_stats"] = 0
    cfg["solver"]["obtain_timings"] = 0
    cfg["solver"]["convergence"] = "RELATIVE_INI"
    cfg["solver"]["tolerance"] = 1e-9
    cfg["solver"]["max_iters"] = 1000
    if "preconditioner" in cfg:
        cfg["preconditioner"]["print_grid_stats"] = 0
        cfg["preconditioner"]["print_solve_stats"] = 0
    if "preconditioner" in cfg["solver"]:
        cfg["solver"]["preconditioner"]["print_grid_stats"] = 0
        cfg["solver"]["preconditioner"]["print_solve_stats"] = 0
    return cfg

In [6]:
configs = {}
for filename in tqdm.tqdm(os.listdir("/workspace/AMGX/src/configs/")):
    if filename.endswith(".json"):
        with open(f"/workspace/AMGX/src/configs/{filename}") as f:
            cfg_json = patch_config(json.load(f))
            name = filename[:-5]
            if "GMRES" in cfg_json["solver"]["solver"]:
                cfg_json["solver"]["solver"] = "PCG"
                name += "__MOD_PCG"
            configs[name] = cfg_json

100%|██████████| 63/63 [00:00<00:00, 37013.75it/s]


In [7]:
double_precision_discrete_problem = discretize(
    continuous_coefficient_3d.problem, all_meshes[3]
)

In [8]:
results = []
for name, cfg_json in tqdm.tqdm(configs.items()):
    cfg = pyamgx.Config().create_from_dict(cfg_json)
    results.append(
        {
            **test_config(
                cfg, double_precision_discrete_problem, rep_setup=1, rep_solve=1
            ),
            "config": name,
        }
    )
    cfg.destroy()

df = pd.DataFrame(results)

 92%|█████████▏| 57/62 [02:03<00:07,  1.44s/it]!!! detected some memory leaks in the code: trying to free non-empty temporary device pool !!!
ptr:     0x7a801c2fd000 size: 4096
100%|██████████| 62/62 [02:05<00:00,  2.03s/it]


In [9]:
df

,setup_time,solve_time,residual,config
0,0.028822,3.424136,NaN,AGGREGATION_DILU
1,0.009472,0.116154,3.924485e-09,AGGREGATION_GS
2,0.006335,0.400870,NaN,AGGREGATION_JACOBI
3,0.005801,0.394770,NaN,AGGREGATION_LOW_DEG_BJ
4,0.018017,3.402673,NaN,AGGREGATION_LOW_DEG_DILU
...,...,...,...,...
57,0.015060,0.102967,1.877222e-09,V-cheby-smoother
58,0.075841,0.071495,2.242131e-09,V-cheby_poly-smoother
59,0.010140,0.418385,NaN,V
60,0.010681,0.758942,NaN,W


In [10]:
working_configs = list(df[df.residual < 1e-6]["config"])
working_configs

['AGGREGATION_GS',
 'AGGREGATION_LOW_DEG_GS',
 'AGGREGATION_THRUST_GS',
 'AMG_AGGRREGATION_CG',
 'AMG_CLASSICAL_AGGRESSIVE_CHEB_L1_TRUNC',
 'AMG_CLASSICAL_AGGRESSIVE_L1__MOD_PCG',
 'AMG_CLASSICAL_AGGRESSIVE_L1_TRUNC__MOD_PCG',
 'AMG_CLASSICAL_L1_AGGRESSIVE_HMIS__MOD_PCG',
 'AMG_CLASSICAL_L1_TRUNC__MOD_PCG',
 'AMG_CLASSICAL_PMIS__MOD_PCG',
 'CG_DILU',
 'FGMRES_CLASSICAL_AGGRESSIVE_HMIS__MOD_PCG',
 'FGMRES_CLASSICAL_AGGRESSIVE_PMIS__MOD_PCG',
 'FGMRES_NOPREC__MOD_PCG',
 'GMRES__MOD_PCG',
 'GMRES_AMG_D2__MOD_PCG',
 'PBICGSTAB_NOPREC',
 'PCG_NOPREC',
 'V-cheby-aggres-L1-trunc-userLambda',
 'V-cheby-aggres-L1-trunc',
 'V-cheby-smoother',
 'V-cheby_poly-smoother',
 'agg_cheb4']

In [11]:
double_precision_discrete_problem = discretize(
    continuous_coefficient_3d.problem, all_meshes[5]
)

In [12]:
results = []
for config in tqdm.tqdm(working_configs):
    cfg_json = configs[config]
    cfg = pyamgx.Config().create_from_dict(cfg_json)
    results.append(
        {
            "config": config,
            **test_config(
                cfg, double_precision_discrete_problem, rep_setup=1, rep_solve=3
            ),
        }
    )

df2 = pd.DataFrame(results)

 91%|█████████▏| 21/23 [00:55<00:08,  4.31s/it]!!! detected some memory leaks in the code: trying to free non-empty temporary device pool !!!
ptr:     0x7a80283c9000 size: 4096
100%|██████████| 23/23 [01:07<00:00,  2.92s/it]


In [13]:
stats5 = df2.pivot_table(
    index="config", values=["setup_time", "solve_time", "residual"], aggfunc="min"
)
stats5

,residual,setup_time,solve_time
config,,,
AGGREGATION_GS,4.428876e-10,0.035187,1.341994
AGGREGATION_LOW_DEG_GS,4.428876e-10,0.033041,1.337558
AGGREGATION_THRUST_GS,5.980847e-10,0.037971,1.156750
AMG_AGGRREGATION_CG,3.603946e-10,0.014994,4.993558
AMG_CLASSICAL_AGGRESSIVE_CHEB_L1_TRUNC,2.166879e-10,0.049103,0.081804
AMG_CLASSICAL_AGGRESSIVE_L1_TRUNC__MOD_PCG,3.652038e-10,0.049177,0.065483
AMG_CLASSICAL_AGGRESSIVE_L1__MOD_PCG,3.104022e-10,0.050933,0.068252
AMG_CLASSICAL_L1_AGGRESSIVE_HMIS__MOD_PCG,3.449247e-10,3.012156,0.067943
AMG_CLASSICAL_L1_TRUNC__MOD_PCG,3.435473e-10,0.079297,0.077044


In [14]:
fast_solvers = list(
    stats5[
        (stats5.solve_time < 2 * stats5.solve_time.min()) & (stats5.residual < 1e-06)
    ].index
)
fast_solvers

['AMG_CLASSICAL_AGGRESSIVE_CHEB_L1_TRUNC',
 'AMG_CLASSICAL_AGGRESSIVE_L1_TRUNC__MOD_PCG',
 'AMG_CLASSICAL_AGGRESSIVE_L1__MOD_PCG',
 'AMG_CLASSICAL_L1_AGGRESSIVE_HMIS__MOD_PCG',
 'AMG_CLASSICAL_L1_TRUNC__MOD_PCG',
 'AMG_CLASSICAL_PMIS__MOD_PCG',
 'FGMRES_CLASSICAL_AGGRESSIVE_HMIS__MOD_PCG',
 'FGMRES_CLASSICAL_AGGRESSIVE_PMIS__MOD_PCG',
 'GMRES_AMG_D2__MOD_PCG']

In [15]:
double_precision_discrete_problem = discretize(
    continuous_coefficient_3d.problem, all_meshes[7]
)

In [16]:
# HMIS Thrust errors for large problems
fast_solvers = [solver for solver in fast_solvers if not "HMIS" in solver]
fast_solvers

['AMG_CLASSICAL_AGGRESSIVE_CHEB_L1_TRUNC',
 'AMG_CLASSICAL_AGGRESSIVE_L1_TRUNC__MOD_PCG',
 'AMG_CLASSICAL_AGGRESSIVE_L1__MOD_PCG',
 'AMG_CLASSICAL_L1_TRUNC__MOD_PCG',
 'AMG_CLASSICAL_PMIS__MOD_PCG',
 'FGMRES_CLASSICAL_AGGRESSIVE_PMIS__MOD_PCG',
 'GMRES_AMG_D2__MOD_PCG']

In [36]:
try:
    results = []
    for config in tqdm.tqdm(fast_solvers):
        cfg_json = configs[config]
        cfg = pyamgx.Config().create_from_dict(cfg_json)
        results.append(
            {
                "config": config,
                **test_config(
                    cfg, double_precision_discrete_problem, rep_setup=3, rep_solve=30
                ),
            }
        )

    df3 = pd.DataFrame(results)
except Exception as e:
    print("Caught runtime error:", e)

100%|██████████| 7/7 [14:27<00:00, 123.97s/it]


In [37]:
stats7 = df3.pivot_table(
    index="config", values=["setup_time", "solve_time", "residual"], aggfunc="min"
)
stats7

,residual,setup_time,solve_time
config,,,
AMG_CLASSICAL_AGGRESSIVE_CHEB_L1_TRUNC,4.799332e-11,2.386806,4.081405
AMG_CLASSICAL_AGGRESSIVE_L1_TRUNC__MOD_PCG,4.175777e-11,2.380612,3.208643
AMG_CLASSICAL_AGGRESSIVE_L1__MOD_PCG,3.612749e-11,2.463136,2.793538
AMG_CLASSICAL_L1_TRUNC__MOD_PCG,3.948690e-11,4.333989,4.110613
AMG_CLASSICAL_PMIS__MOD_PCG,4.175777e-11,2.377730,3.210254
FGMRES_CLASSICAL_AGGRESSIVE_PMIS__MOD_PCG,4.175777e-11,2.380934,3.209249
GMRES_AMG_D2__MOD_PCG,3.350605e-11,2.249796,5.233399


In [38]:
stats7.sort_values("solve_time")

,residual,setup_time,solve_time
config,,,
AMG_CLASSICAL_AGGRESSIVE_L1__MOD_PCG,3.612749e-11,2.463136,2.793538
AMG_CLASSICAL_AGGRESSIVE_L1_TRUNC__MOD_PCG,4.175777e-11,2.380612,3.208643
FGMRES_CLASSICAL_AGGRESSIVE_PMIS__MOD_PCG,4.175777e-11,2.380934,3.209249
AMG_CLASSICAL_PMIS__MOD_PCG,4.175777e-11,2.377730,3.210254
AMG_CLASSICAL_AGGRESSIVE_CHEB_L1_TRUNC,4.799332e-11,2.386806,4.081405
AMG_CLASSICAL_L1_TRUNC__MOD_PCG,3.948690e-11,4.333989,4.110613
GMRES_AMG_D2__MOD_PCG,3.350605e-11,2.249796,5.233399


In [39]:
stats7.to_csv("../results/amgx_3d.csv")